# Part 3 — Synthetic population preprocessing
Prepare existing MATSim **input population** for comparison with Mikrozensus. No simulation is run and no simulation outputs are read.

Use the same `Boundary/neuchatel.shp` and point-intersection rule as `microcensus_preprocessing.ipynb`. Retain all spatially selected residents, including non-travellers and cross-boundary activities. Export full person attributes, selected-plan activities, legs, trips, person-level chains and audit records to `simulation_filter/`.

**Provisional metadata:** the input declares `Atlantis`, not a usable CRS. EPSG:2056 is an explicit working assumption based on coordinate magnitudes, not verified provenance. The represented weekday is unconfirmed. Exports are therefore provisional until these two assumptions are documented. No invented weekday filter is applied.

In [ ]:
from pathlib import Path
import json, gzip, math, hashlib, platform
import xml.etree.ElementTree as ET
import pandas as pd
import geopandas as gpd
from pyproj import CRS

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Neuchatel_10pct/neuchatel_config.xml').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the project directory or a subdirectory.')
SCENARIO = ROOT / 'Neuchatel_10pct'
POPULATION = next((SCENARIO / name for name in ['neuchatel_population.xml.gz', 'neuchatel_population.xml']
                   if (SCENARIO / name).exists()), None)
assert POPULATION is not None, 'Population input not found.'
BOUNDARY = ROOT / 'Boundary/neuchatel.shp'
EXPORT = ROOT / 'simulation_filter'
EXPORT.mkdir(exist_ok=True)

SOURCE_CRS = 'EPSG:2056'  # Working assumption; confirm with the scenario provider.
CRS_CONFIRMED = False
CRS_EVIDENCE = ''
SCENARIO_DAY_TYPE = 'unconfirmed'  # Set to 'weekday' only with evidence.
DAY_TYPE_EVIDENCE = ''
HOME_FIELDS = ('home_coordinate_x', 'home_coordinate_y')
# Preserve the alternative pair and audit sensitivity; do not silently fall back.
COMPARISON_MIN_AGE = 6  # User-confirmed comparison scope; retain all ages in main exports.
AGE_SCOPE_CONFIRMED = True
AGE_BINS = [0, 6, 18, 30, 45, 65, float('inf')]
AGE_LABELS = ['0–5', '6–17', '18–29', '30–44', '45–64', '65+']
assert SCENARIO_DAY_TYPE in {'unconfirmed', 'weekday', 'weekend', 'mixed'}
if CRS_CONFIRMED: assert CRS_EVIDENCE.strip()
if SCENARIO_DAY_TYPE == 'weekday': assert DAY_TYPE_EVIDENCE.strip()
print('Input:', POPULATION.name)
print('Provisional CRS:', SOURCE_CRS, '| Day type:', SCENARIO_DAY_TYPE)

## 1. Parse each person and exactly one selected plan
Keep original person attributes. Do not use canton labels for selection. Require exactly one selected plan (rather than silently choosing the first). `* interaction` activities are stage activities; preserve them in the activity table but omit them from purpose chains. `outside` and `border` remain explicit because true purposes/locations beyond the cropped scenario may be unavailable.

Times and route distances are input-plan values, not observed simulation results. Preserve raw times; support times beyond 24:00 and leave missing values missing.

In [ ]:
def open_xml(path):
    return gzip.open(path, 'rb') if path.suffix == '.gz' else path.open('rb')

def attributes(element):
    return {a.get('name'): a.text for a in element.findall('./attributes/attribute')}

def number(value):
    try:
        result = float(value)
        return result if math.isfinite(result) else float('nan')
    except (TypeError, ValueError):
        return float('nan')

def seconds(value):
    if value is None or value == 'undefined': return float('nan')
    if ':' not in value: return number(value)
    parts = value.split(':')
    return sum(number(v) * m for v, m in zip(parts, [3600, 60, 1])) if len(parts) == 3 else float('nan')

purpose_map = {'home_secondary': 'home', 'work_secondary': 'work',
               'education_secondary': 'education', 'shop': 'shopping'}
person_rows, activity_rows, leg_rows, trip_rows = [], [], [], []
with open_xml(POPULATION) as stream:
    context = ET.iterparse(stream, events=('start', 'end'))
    _, root_element = next(context)
    for event, element in context:
        if event != 'end' or element.tag != 'person': continue
        pid = element.get('id')
        plans = element.findall('plan')
        chosen = [p for p in plans if p.get('selected') == 'yes']
        assert len(chosen) == 1, f'{pid}: expected exactly one selected plan'
        row = attributes(element)
        assert 'person' not in row
        row.update(person=pid, plan_count=len(plans))
        person_rows.append(row)
        plan_activities, plan_legs = [], []
        for sequence, child in enumerate(chosen[0]):
            if child.tag in {'activity', 'act'}:
                raw = child.get('type', '')
                stage = raw.endswith(' interaction')
                a = dict(person=pid, sequence=sequence, activity_index=len(plan_activities),
                         activity_type_raw=raw, activity_type=purpose_map.get(raw, raw),
                         is_stage=stage, is_boundary_proxy=raw in {'outside', 'border'},
                         x=number(child.get('x')), y=number(child.get('y')),
                         facility=child.get('facility'), link=child.get('link'),
                         start_time=child.get('start_time'), end_time=child.get('end_time'),
                         start_time_s=seconds(child.get('start_time')),
                         end_time_s=seconds(child.get('end_time')),
                         max_duration_s=seconds(child.get('max_dur')))
                a.update({'activity_attr_' + k: v for k, v in attributes(child).items()})
                plan_activities.append(a)
            elif child.tag == 'leg':
                route = child.find('route')
                leg = dict(person=pid, sequence=sequence, leg_index=len(plan_legs),
                           mode=child.get('mode'), routing_mode=attributes(child).get('routingMode'),
                           departure_time_s=seconds(child.get('dep_time')),
                           travel_time_s=seconds(child.get('trav_time')),
                           route_distance_m=number(route.get('distance')) if route is not None else float('nan'))
                plan_legs.append(leg)
        main = [a for a in plan_activities if not a['is_stage']]
        for index, a in enumerate(main): a['main_activity_index'] = index
        for index, (origin, destination) in enumerate(zip(main, main[1:])):
            legs = [l for l in plan_legs if origin['sequence'] < l['sequence'] < destination['sequence']]
            trip_rows.append(dict(person=pid, trip_index=index,
                origin_activity_index=origin['activity_index'], destination_activity_index=destination['activity_index'],
                origin_type=origin['activity_type'], destination_type=destination['activity_type'],
                origin_x=origin['x'], origin_y=origin['y'], destination_x=destination['x'], destination_y=destination['y'],
                leg_count=len(legs), modes='|'.join(l['mode'] or '' for l in legs),
                routing_modes='|'.join(dict.fromkeys(l['routing_mode'] for l in legs if l['routing_mode'])),
                route_distance_m=sum(l['route_distance_m'] for l in legs) if legs else float('nan'),
                has_boundary_proxy=origin['is_boundary_proxy'] or destination['is_boundary_proxy']))
        row['activity_chain_raw'] = ' > '.join(a['activity_type_raw'] for a in main)
        row['activity_chain'] = ' > '.join(a['activity_type'] for a in main)
        row['main_activity_count'] = len(main)
        row['trip_count'] = max(len(main) - 1, 0)
        row['leg_count'] = len(plan_legs)
        # A home→home trip is not a stay-home day. Only a single primary-home activity without legs qualifies.
        stay = len(main) == 1 and main[0]['activity_type_raw'] == 'home' and not plan_legs
        row['day_status'] = 'at home all day' if stay else ('travelled' if len(main) > 1 and plan_legs else 'unknown / check')
        row['at_home_all_day'] = stay
        row['has_boundary_proxy'] = any(a['is_boundary_proxy'] for a in main)
        activity_rows.extend(plan_activities)
        leg_rows.extend(plan_legs)
        element.clear()
        root_element.remove(element)
    declared_crs = attributes(root_element).get('coordinateReferenceSystem')
all_persons = pd.DataFrame(person_rows)
all_activities = pd.DataFrame(activity_rows)
all_legs = pd.DataFrame(leg_rows)
all_trips = pd.DataFrame(trip_rows)
assert all_persons.person.is_unique
print('Input persons:', len(all_persons), '| Declared CRS:', declared_crs)

## 2. Same home-location boundary filter as Mikrozensus
`home_coordinate_x/y` is the explicit primary choice; `home_x/y` is retained and separately tested. Coordinate validity and different inclusion decisions are exported for review. The CRS assumption must be confirmed before treating spatial results as final.

In [ ]:
boundary = gpd.read_file(BOUNDARY)
assert boundary.crs is not None
assert boundary.geometry.notna().all() and boundary.geometry.is_valid.all()
shape = boundary.geometry.union_all()

def locate(frame, xcol, ycol):
    x, y = pd.to_numeric(frame[xcol], errors='coerce'), pd.to_numeric(frame[ycol], errors='coerce')
    valid = x.notna() & y.notna() & x.abs().lt(float('inf')) & y.abs().lt(float('inf'))
    if CRS.from_user_input(SOURCE_CRS).to_epsg() == 2056:
        valid &= x.between(2_000_000, 3_000_000) & y.between(800_000, 1_500_000)
    points = gpd.GeoDataFrame(index=frame.index[valid], geometry=gpd.points_from_xy(x[valid], y[valid]), crs=SOURCE_CRS).to_crs(boundary.crs)
    inside = pd.Series(pd.NA, index=frame.index, dtype='boolean')
    inside.loc[valid] = points.geometry.intersects(shape)
    bx, by = pd.Series(float('nan'), index=frame.index), pd.Series(float('nan'), index=frame.index)
    bx.loc[valid], by.loc[valid] = points.geometry.x, points.geometry.y
    return valid, inside, bx, by

valid, inside, bx, by = locate(all_persons, *HOME_FIELDS)
alt_valid, alt_inside, _, _ = locate(all_persons, 'home_x', 'home_y')
all_persons['home_coordinate_valid'] = valid
all_persons['home_inside_boundary'] = inside
all_persons['home_boundary_x'], all_persons['home_boundary_y'] = bx, by
all_persons['alternate_home_inside_boundary'] = alt_inside
all_persons['home_pair_distance_m'] = (
    (pd.to_numeric(all_persons[HOME_FIELDS[0]], errors='coerce') - pd.to_numeric(all_persons.home_x, errors='coerce')) ** 2 +
    (pd.to_numeric(all_persons[HOME_FIELDS[1]], errors='coerce') - pd.to_numeric(all_persons.home_y, errors='coerce')) ** 2
) ** 0.5
all_persons['home_filter_disagrees'] = inside.ne(alt_inside).fillna(False)
home_audit = all_persons[['person', 'cantonName', *HOME_FIELDS, 'home_x', 'home_y',
                         'home_coordinate_valid', 'home_inside_boundary', 'alternate_home_inside_boundary',
                         'home_pair_distance_m', 'home_filter_disagrees']].copy()
persons = all_persons.loc[inside.fillna(False)].copy()
assert len(persons) > 0
selected_ids = set(persons.person)
activities = all_activities.loc[all_activities.person.isin(selected_ids)].copy()
legs = all_legs.loc[all_legs.person.isin(selected_ids)].copy()
trips = all_trips.loc[all_trips.person.isin(selected_ids)].copy()
print('Spatially selected residents:', len(persons))
print('Home-pair inclusion disagreements:', int(all_persons.home_filter_disagrees.sum()))
print('Invalid primary home coordinates:', int((~valid).sum()))

## 3. Analysis-ready fields without discarding residents
Candidate demographic variables: age, sex, employment. Candidate spatial contexts: `municipalityType`, `ovgk`; their survey crosswalks still require verification. Keep raw values including literal `None` for public-transport accessibility.

No age exclusion is applied to the main exports. `comparison_age_eligible` identifies the user-confirmed age-6+ comparison subset; do not truncate at the maximum observed survey age. The age-group field supports stratified activity analyses. MATSim sample weights are 1 for within-sample distributions, not population expansion weights and not survey `WP` weights.

In [ ]:
persons['canton_label_needs_review'] = persons.cantonName.ne('neuchatel')
persons['age_numeric'] = pd.to_numeric(persons.age, errors='coerce')
persons['age_group'] = pd.cut(persons.age_numeric, bins=AGE_BINS, labels=AGE_LABELS, right=False)
persons['sex_group'] = persons.sex.map({'f': 'female', 'm': 'male'}).fillna('unknown / check')
persons['employment_group'] = persons.employed.map({'true': 'employed', 'false': 'not employed'}).fillna('unknown / check')
persons['comparison_age_eligible'] = persons.age_numeric.ge(COMPARISON_MIN_AGE)
persons['age_valid'] = persons.age_numeric.between(0, 120)
persons['sample_weight'] = 1.0
persons['scenario_day_type'] = SCENARIO_DAY_TYPE
persons['weekday_verified'] = SCENARIO_DAY_TYPE == 'weekday'
persons['spatial_crs_verified'] = CRS_CONFIRMED

av, ai, ax, ay = locate(activities, 'x', 'y')
activities['coordinate_valid'] = av
activities['inside_boundary'] = ai
activities['boundary_x'], activities['boundary_y'] = ax, ay
activities['spatial_analysis_eligible'] = av & ~activities.is_stage & ~activities.is_boundary_proxy
activities['is_home_activity'] = activities.activity_type.eq('home')
# Keep all rows. Filter is_stage=False for purpose analysis; spatial_analysis_eligible for maps.
context_cols = ['person', 'age_group', 'sex_group', 'employment_group', 'comparison_age_eligible', 'sample_weight']
activities = activities.merge(persons[context_cols], on='person', how='left', validate='many_to_one')
trips = trips.merge(persons[context_cols], on='person', how='left', validate='many_to_one')
chains = persons[['person', 'activity_chain_raw', 'activity_chain', 'day_status', 'at_home_all_day',
                 'main_activity_count', 'trip_count', 'has_boundary_proxy', *context_cols[1:]]].copy()
print(persons.day_status.value_counts().to_string())
print('Age-eligible residents (age >= 6):', int(persons.comparison_age_eligible.sum()))

## 4. Validate and export
CSV outputs are UTF-8, comma-separated, without dataframe indices; join by `person` (read it as a string). For activity and leg ordering use `sequence`; `trip_index` describes successive non-stage activities. Raw activities and legs are not interchangeable with trips.

No trajectories are removed because they cross the boundary. `outside` / `border` proxy locations are explicitly excluded by the spatial-eligibility flag, not deleted. Exact activity timing and time-budget comparisons require a separate consistency review.

In [ ]:
assert persons.person.is_unique
assert not activities.duplicated(['person', 'activity_index']).any()
assert not legs.duplicated(['person', 'leg_index']).any()
assert not trips.duplicated(['person', 'trip_index']).any()
for frame in [activities, legs, trips, chains]: assert frame.person.isin(persons.person).all()
counts = trips.groupby('person').size().reindex(persons.person, fill_value=0).to_numpy()
assert (counts == persons.trip_count.to_numpy()).all()
assert persons.trip_count.sum() == len(trips)
stay_ids = set(persons.loc[persons.at_home_all_day, 'person'])
assert not trips.person.isin(stay_ids).any()
assert not legs.person.isin(stay_ids).any()
assert activities.loc[activities.person.isin(stay_ids)].groupby('person').size().eq(1).all()
assert set(chains.person) == selected_ids

review = persons.loc[persons.day_status.eq('unknown / check') | ~persons.age_valid | persons.home_filter_disagrees | persons.canton_label_needs_review].copy()
audit = pd.DataFrame([
    ('input_persons', len(all_persons)), ('invalid_home_coordinates', int((~valid).sum())),
    ('residents_inside_boundary', len(persons)), ('home_filter_disagreements_all_persons', int(all_persons.home_filter_disagrees.sum())),
    ('age_eligible_residents', int(persons.comparison_age_eligible.sum())),
    ('at_home_all_day', len(stay_ids)), ('review_persons', len(review)),
    ('selected_plan_activities_including_stages', len(activities)),
    ('non_stage_activities', int((~activities.is_stage).sum())),
    ('trips', len(trips)), ('legs', len(legs)),
    ('invalid_activity_coordinates', int((~activities.coordinate_valid).sum()))
], columns=['stage', 'count'])
tables = {'persons': persons, 'activities': activities, 'legs': legs, 'trips': trips,
          'activity_chains': chains, 'home_filter_audit': home_audit, 'review_persons': review, 'filter_audit': audit}
for name, frame in tables.items():
    path = EXPORT / f'simulation_{name}.csv'
    frame.to_csv(path, index=False, encoding='utf-8')
    restored = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert len(restored) == len(frame) and restored.columns.tolist() == frame.columns.tolist()
    if 'person' in frame: assert restored.person.tolist() == frame.person.astype(str).tolist()

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

metadata = dict(population_file=str(POPULATION.relative_to(ROOT)), population_sha256=sha256(POPULATION),
    boundary_file=str(BOUNDARY.relative_to(ROOT)),
    boundary_sha256={p.name: sha256(p) for p in BOUNDARY.parent.glob('neuchatel.*')},
    declared_crs=declared_crs, assumed_source_crs=SOURCE_CRS, crs_confirmed=CRS_CONFIRMED,
    crs_evidence=CRS_EVIDENCE, boundary_crs=boundary.crs.to_wkt(),
    home_fields=list(HOME_FIELDS), home_rule='intersects; includes boundary points; no canton filter',
    scenario_day_type=SCENARIO_DAY_TYPE, day_type_evidence=DAY_TYPE_EVIDENCE,
    weekday_filter_verified=SCENARIO_DAY_TYPE == 'weekday',
    comparison_min_age=COMPARISON_MIN_AGE, age_scope_confirmed=AGE_SCOPE_CONFIRMED, age_scope_basis="User-approved age >= 6; not independent survey-universe verification",
    comparison_weights={"Mikrozensus": "WP", "MATSim": "unit sample counts"},
    provisional=not (CRS_CONFIRMED and SCENARIO_DAY_TYPE == 'weekday' and AGE_SCOPE_CONFIRMED),
    purpose_mapping=purpose_map, selected_plan_only=True,
    python_version=platform.python_version(), pandas_version=pd.__version__, geopandas_version=gpd.__version__,
    counts=dict(audit.itertuples(index=False, name=None)))
(EXPORT / 'preprocessing_metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print(audit.to_string(index=False))
print('All export checks passed. Provisional:', metadata['provisional'])

readme = """# MATSim Part 3 preprocessing outputs

Generated by `../simulation_preprocessing.ipynb` from the input population, without running a simulation.

## Status
These outputs are provisional. The source declares CRS `Atlantis`; EPSG:2056 is an explicit working assumption. The scenario's weekday requires source confirmation. The user has confirmed age >= 6 as the comparison scope; younger MATSim residents remain in the main exports. See preprocessing_metadata.json. Change the notebook configuration and rerun when evidence is available.

## Files
- simulation_persons.csv: one row per spatially selected resident; all original attributes, harmonized demographic candidates, home coordinates, stay-home status and age eligibility.
- simulation_activities.csv: all selected-plan activities including stage activities; raw and normalized purposes, coordinates, spatial flags, age/sex/employment groups.
- simulation_legs.csv: individual transport legs, modes and input-plan times/distances.
- simulation_trips.csv: transitions between consecutive non-stage activities; constituent leg modes and endpoint coordinates. Outside/border proxies remain explicit.
- simulation_activity_chains.csv: one raw and normalized chain per person, including home-only people and demographic groups.
- simulation_home_filter_audit.csv: every input person, coordinate validity and primary/alternate home-boundary inclusion.
- simulation_review_persons.csv: selected residents with unresolved day status, invalid age, unexpected canton labels or differing home-filter decisions; may be empty.
- simulation_filter_audit.csv: counts through the workflow.
- preprocessing_metadata.json: assumptions, versions, input checksums, boundary CRS and counts.

## Read and compare
CSV files use UTF-8 and commas. Read `person` as a string. Use `keep_default_na=False` if preserving literal `None` accessibility categories; convert numeric columns explicitly as needed. Join tables on `person`. Sequence and activity/leg/trip indices are zero-based.

Use persons for demographic denominators. Do not drop home-only people by inner-joining trips. `sample_weight=1` is a within-sample weight, not an expansion factor. Use Mikrozensus WP weights and MATSim unit sample weights for proportions within the same eligible sample. Cross-source category mappings still require verification.

For purpose chains exclude stage activities. The normalized map merges home_secondary, work_secondary and education_secondary into their primary categories and shop into shopping; raw labels remain available. It does not collapse consecutive equal purposes. A home-to-home trip is travel, not an all-day-home observation. Only a single primary-home activity with no legs is classified as at home all day.

For activity maps use spatial_analysis_eligible=True (valid coordinates, non-stage, no outside/border proxies). Inside-boundary is a separate flag: cross-boundary activities are retained. For home maps use one home point per person. Geographic comparison requires a shared CRS and spatial aggregation. Grid/region definitions and normalization are downstream analysis choices.

The primary home fields are home_coordinate_x/y. home_x/y are retained and compared in the audit. Invalid coordinates are not imputed. Broad Swiss-coordinate plausibility bounds are a quality screen, not a substitute for polygon intersection. No canton-label filter is used.

All ages remain in main exports. comparison_age_eligible implements the user-approved minimum age of 6 for formal comparisons; do not impose the maximum observed survey age as an upper cutoff. No individual weekday is fabricated. Input-plan timing has not been validated as an observed daily time budget.
"""
(EXPORT / 'README.md').write_text(readme, encoding='utf-8')


## 5. How these outputs support Part 3

| Required analysis | Output and fields |
|---|---|
| Three demographics | `simulation_persons.csv`: `age_group`, `sex_group`, `employment_group`; raw attributes retained |
| Two spatial contexts | Persons: `municipalityType`, `ovgk`, home coordinates; verify Mikrozensus category crosswalks |
| Aggregate and demographic-stratified chains | `simulation_activity_chains.csv`: raw/normalized chain, `age_group`, stay-home status |
| Activity maps and demographic-stratified spatial differences | `simulation_activities.csv`: non-stage activities, transformed coordinates, purposes, `age_group` |
| Optional trip-level analysis | `simulation_trips.csv`: endpoints, purposes, modes; no inferred main-mode classification |
| Quality control | Home audit, review persons, filter audit, metadata |

Before final comparison: confirm source CRS and scenario weekday; apply the same age groups and documented category mappings to Mikrozensus; use survey `WP` appropriately; choose a common spatial grid and comparison denominator. Stage activities must not count as destinations. Home-only people must remain in person-level denominators. For home-location distributions use one home point per person, not repeated home activities.

All residents are preserved in the main files. Use `comparison_age_eligible` consistently across person/activity/chain tables for all formal comparisons (user-approved age >= 6). There is no upper-age cutoff. Tables remain provisional until metadata requirements are resolved.